# Step 07.5 — Token + Position Addition

现在我们已经有两种信息：

```text
token embedding    -> 这个 token 是什么
position embedding -> 这个 token 在哪里
```

早期 GPT 风格的做法非常直接：

$$
h_{b,t}=e_{token_{b,t}}+p_t
$$

也就是：

```text
(B,T,C) + (T,C) -> (B,T,C)
```

这一节完整解决 5 个问题：

1. broadcasting 到底怎样把 `(T,C)` 加到 `(B,T,C)`；
2. 同一个 token 为什么会因 position 不同而得到不同初始 hidden state；
3. 为什么这里常用 addition，而不是 concatenation；
4. 两种信息相加后，后续网络为什么仍然可以利用它们，以及 addition 的真实代价；
5. 用 heatmap 看 token / position / sum，并完成 Step 7 的整条 representation pipeline。

## 1. `(B,T,C) + (T,C)` 的 Broadcasting 到底发生了什么？

假设：

```text
B = 2
T = 3
C = 4
```

token embedding：

```text
tok.shape = (2,3,4)
```

position embedding：

```text
pos.shape = (3,4)
```

根据 broadcasting 从右往左比较 shape：

```text
(B,T,C)
   (T,C)
```

`(T,C)` 可以看成前面缺了一个 size-1 维度：

```text
(1,T,C)
```

然后这一个 `1` 沿 Batch axis 逻辑扩展成 B：

```text
(1,T,C)
   ↓ broadcast over B
(B,T,C)
```

因此同一套 position vectors 会加到 batch 中每一条 sequence。

In [ ]:
import mlx.core as mx
import mlx.nn as nn

B, T, V, C = 2, 3, 5, 4

X = mx.array([
    [1, 2, 1],
    [3, 1, 4],
])

wte = nn.Embedding(V, C)
wpe = nn.Embedding(8, C)

tok = wte(X)              # (B,T,C)
positions = mx.arange(T) # (T,)
pos = wpe(positions)      # (T,C)

h = tok + pos

mx.eval(tok, pos, h)

print("tok shape:", tok.shape)
print("pos shape:", pos.shape)
print("h shape  :", h.shape)

可以把 broadcasting 写得更显式：

```python
pos_explicit = pos[None, :, :]
```

shape 变成：

```text
(1,T,C)
```

然后：

```python
tok + pos_explicit
```

与直接 `tok + pos` 完全等价。

In [ ]:
pos_explicit = pos[None, :, :]
h_explicit = tok + pos_explicit

mx.eval(h_explicit)

print("pos_explicit shape:", pos_explicit.shape)
print("same result:", bool(mx.allclose(h, h_explicit)))

# 再检查一个具体位置
b, t = 1, 2
manual = tok[b, t, :] + pos[t, :]
print("h[b,t,:] == tok[b,t,:] + pos[t,:] ?",
      bool(mx.allclose(h[b, t, :], manual)))

## 2. 同一个 Token + 不同 Position = 不同初始 Hidden State

假设 sequence 是：

```text
A A A
```

token embedding 会得到：

```text
e_A
e_A
e_A
```

完全一样。

但 position vectors 是：

```text
p_0
p_1
p_2
```

所以相加以后：

```text
h_0 = e_A + p_0
h_1 = e_A + p_1
h_2 = e_A + p_2
```

同一个 token 因为位置不同，初始 hidden representation 也不同。

In [ ]:
same_tokens = mx.array([[1, 1, 1]])  # B=1, T=3

tok_same = wte(same_tokens)
pos_same = wpe(mx.arange(3))
h_same = tok_same + pos_same

mx.eval(tok_same, pos_same, h_same)

print("token embeddings pos0/pos1 equal:",
      bool(mx.allclose(tok_same[0, 0], tok_same[0, 1])))
print("final h pos0/pos1 equal:",
      bool(mx.allclose(h_same[0, 0], h_same[0, 1])))

反过来，如果两个 batch sample 在**同一个 position**上恰好有同一个 token，那么它们的这一步初始表示也相同：

```text
same token identity
+ same position identity
= same initial hidden vector
```

后面一旦经过 Attention，因为两条 sequence 的上下文不同，它们的 contextual hidden states 就可以分化。

## 3. 为什么选择 Addition，而不是 Concatenation？

理论上并不是只能相加。

另一种思路可以是：

```text
[token vector ; position vector]
```

也就是 concatenate。

如果 token 和 position 都是 C 维：

```text
token:    (B,T,C)
position: (B,T,C)
```

concatenate 后：

```text
(B,T,2C)
```

这会把整个 model width 翻倍。

In [ ]:
pos_batched = mx.broadcast_to(pos[None, :, :], (B, T, C))
concat = mx.concatenate([tok, pos_batched], axis=-1)

print("tok        :", tok.shape)
print("pos_batched:", pos_batched.shape)
print("addition   :", h.shape)
print("concat     :", concat.shape)

如果后面的 Transformer 仍想维持 model width C，concatenation 后通常还得再投影回来：

```text
2C -> C
```

例如一个无 bias Linear 的参数量就是：

$$
C\times2C=2C^2
$$

而 addition：

```text
(B,T,C) + (T,C)
-> (B,T,C)
```

直接保留原 model width，不需要额外 projection。

In [ ]:
C_demo = 128
projection_params = (2 * C_demo) * C_demo

print("If concat gives 2C and we project 2C->C:")
print("extra Linear parameters (bias=False):", projection_params)

所以 early GPT 使用 addition 的重要工程优势是：

- 保持固定 `(B,T,C)` 接口；
- 不把后续所有 Attention / MLP 的宽度直接翻倍；
- 计算图简单；
- token / position 两套 embedding 可以联合训练到同一个 hidden space。

但要准确：

> **Concatenation 并不是数学上错误。**

它只是另一种架构选择，通常需要更多宽度或额外 projection。Addition 是一种高效、经典的设计取舍。

## 4. 相加以后信息不会“混没了吗”？——能利用，但不是无损分解

这是非常自然的问题。

如果：

```text
h = e + p
```

只给你 `h`，一般不能唯一恢复原来的 `e` 和 `p`。

因为很多不同 pair 都可能得到同一个和：

```text
[1,2] + [3,4] = [4,6]
[2,1] + [2,5] = [4,6]
```

所以从纯数学映射 `(e,p) -> e+p` 来看，它是多对一的，并不是无损编码。

### 那模型为什么还能用？

因为 `wte`、`wpe` 和后续网络不是固定后再硬拼起来，而是**一起为了同一个训练 loss 联合优化**。

模型不需要显式执行：

```text
把 h 精确拆回 token vector + position vector
```

它只需要让合成后的 h 对后续预测足够有用。

更重要的是，对任意后续 Linear transformation W：

$$
W(e+p)=We+Wp
$$

所以一个线性层能够对 token contribution 和 position contribution 分别产生作用，再在输出中叠加。

In [ ]:
# 用 row-vector convention 验证：(e+p) @ W.T = e@W.T + p@W.T
e = mx.array([1.0, 2.0, -1.0])
p = mx.array([0.5, -0.5, 1.0])
W = mx.array([
    [1.0, 0.0, 2.0],
    [0.0, 1.0, -1.0],
])

left = (e + p) @ W.T
right = e @ W.T + p @ W.T

mx.eval(left, right)
print("W(e+p):", left)
print("We+Wp :", right)
print("same  :", bool(mx.allclose(left, right)))

后面再经过 nonlinearities、Attention、多层 Block，模型可以学习复杂得多的组合方式。

但仍然不要过度宣传：

- addition 的确把两份 C 维信息压进一份 C 维状态；
- 理论上可能出现 collision；
- 它不能保证 token / position 永远可完美分离；
- 实际效果来自联合训练与整个网络的表达能力。

因此更准确的说法是：**addition 是保持固定宽度的高效融合方式，而不是免费的无损合并。**

## 5. 用 Heatmap 直接看：Token + Position = Initial Hidden State

对一个 sample：

```text
token matrix    : (T,C)
position matrix : (T,C)
sum             : (T,C)
```

非常适合并排画 heatmap。

绘图时有一个重要技巧：如果要比较多个 heatmap，最好使用相同的 `vmin/vmax`，否则每张图会自动使用不同颜色尺度，看起来可能误导。

In [ ]:
import matplotlib.pyplot as plt

sample_tok = tok[0]
sample_pos = pos
sample_sum = h[0]

mx.eval(sample_tok, sample_pos, sample_sum)

tok_plot = sample_tok.tolist()
pos_plot = sample_pos.tolist()
sum_plot = sample_sum.tolist()

all_values = [v for row in tok_plot + pos_plot + sum_plot for v in row]
vmin = min(all_values)
vmax = max(all_values)

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)

for ax, matrix, title in zip(
    axes,
    [tok_plot, pos_plot, sum_plot],
    ["Token embedding", "Position embedding", "Token + position"],
):
    image = ax.imshow(matrix, aspect="auto", vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_xlabel("feature c")

axes[0].set_ylabel("position t")
fig.colorbar(image, ax=axes.ravel().tolist(), label="value")
plt.show()

现在 Step 7 的 representation pipeline 已经完整：

```text
raw text
  ↓ tokenizer
token IDs (B,T)
  ↓ wte
token embeddings (B,T,C)

positions [0...T-1]
  ↓ wpe
position embeddings (T,C)

token + position
  ↓ broadcasting
initial hidden state (B,T,C)
```

这个 `(B,T,C)` 才是后面 sequence model 真正接手的连续表示。

### Learned Absolute Addition 的边界

当前方案仍有前面提过的限制：

- wpe 只定义到 `block_size-1`；
- absolute positions 不直接编码相对距离；
- addition 把 token / position 压进同一个 C 维空间；
- 现代 LLM 常改用 RoPE、relative position bias 等其它设计。

但作为从零 MiniGPT，它有一个巨大的教学优势：**每一步都可以直接看到和手算。**

### Step 7 最终结论

- `(T,C)` position embeddings 在加到 `(B,T,C)` token embeddings 时，逻辑上先视为 `(1,T,C)`，再沿 B broadcasting；
- 每个位置满足 `h[b,t,:] = tok[b,t,:] + pos[t,:]`；
- 同一 token 在不同位置因为加入不同 `p_t` 而获得不同初始 hidden representation；
- addition 保持 model width C 不变，而 concatenation 会变成 2C，通常还需额外 projection 或更宽的后续网络；
- concatenation 不是错误，只是不同的架构取舍；
- addition 本身不是无损分解：从 `e+p` 一般不能唯一还原 e、p；
- token/position embeddings 与后续网络联合训练，使合成后的 C 维状态对任务有用；
- Linear 满足 `W(e+p)=We+Wp`，说明后续线性变换可以分别响应两部分 contribution；
- 完成本节后 Step 7 完成。
- 下一步进入 Step 8.1：回到历史发展线，解释普通 MLP 为什么不自然适合 sequence，以及 RNN 为什么会被发明出来。